<a href="https://colab.research.google.com/github/DCOMP-UFS/Engenharia_SoftwareII_2025-2_T04_screenpipe/blob/main/Modelo_1_bge_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


In [1]:
# 1. Instalar as bibliotecas necessárias
!pip install transformers accelerate bitsandbytes sentence-transformers Pillow datasets
!pip install -q git+https://github.com/huggingface/peft.git  # PEFT é útil para modelos grandes

# 2. Clonar o repositório ScreenPipe
!git clone https://github.com/mediar-ai/screenpipe.git
%cd screenpipe
!ls # Verifique os diretórios clonados

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'screenpipe'...
remote: Enumerating objects: 39856, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 39856 (delta 0), reused 0 (delta 0), pack-reused 39851 (from 2)
Receiving objects: 100% (39856/39856), 320.02 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (24962/24962), done.
Filtering content: 100% (18/18), 71.84 MiB | 2.25 MiB/s, done.
/content/screenpipe
Cargo.lock	 README-ja.md	       screenpipe-events
Cargo.toml	 README.md	       screenpipe-integrations
content		 README-zh_CN.md       screenpipe-js
CONTRIBUTING.md  rust-toolchain.toml   screenpipe-server
install.ps1	 screenpipe-app-tauri  screenpipe-vision
install.sh	 screenpipe-audio      TESTING.md
LICENSE.md	 screenpipe-core
pipes		 screenpi

In [2]:
!pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

# Modelo 1: BGE-base para embeddings
embed_model_name = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(embed_model_name, device="cuda")

print("Modelo de embeddings carregado:", embed_model_name)


In [ ]:
import os

%cd /content/screenpipe

# extensões que queremos indexar
EXTENSOES_VALIDAS = (".rs", ".ts", ".tsx", ".js", ".md", ".toml", ".yaml", ".yml")

docs = []   # cada item: {"texto": ..., "arquivo": ..., "modulo": ...}

for root, dirs, files in os.walk("."):
    for fname in files:
        if fname.endswith(EXTENSOES_VALIDAS):
            path = os.path.join(root, fname)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
            except Exception:
                continue

            # módulo = nome da primeira pasta depois de ./  (ex: ./screenpipe-core/src/... -> screenpipe-core)
            rel = path.lstrip("./")
            modulo = rel.split(os.sep)[0] if os.sep in rel else rel

            # dividir em chunks de ~1000 caracteres
            CHUNK_SIZE = 1000
            for i in range(0, len(text), CHUNK_SIZE):
                chunk = text[i:i+CHUNK_SIZE]
                if chunk.strip():
                    docs.append({
                        "texto": chunk,
                        "arquivo": path,
                        "modulo": modulo,
                        "offset": i,  # onde começa no arquivo
                    })

print(f"Total de chunks criados: {len(docs)}")
print("Exemplo de doc:", docs[0]["arquivo"], "| módulo:", docs[0]["modulo"])


In [ ]:
embeddings[0][0]

In [ ]:
import numpy as np
from sentence_transformers.util import cos_sim

def buscar_trechos(query, modulo=None, top_k=5):
    """
    query: texto da busca (ex: 'captura de tela', 'API de busca', 'pipes/plugins')
    modulo: se quiser filtrar por 'screenpipe-core', 'screenpipe-server', etc.
    top_k: quantos trechos retornar
    """
    # embedding da consulta
    q_emb = embed_model.encode([query], normalize_embeddings=True)[0]

    # similaridade com todos os chunks
    scores = cos_sim(q_emb, embeddings)[0].cpu().numpy()

    # se quiser filtrar por módulo:
    indices_validos = list(range(len(docs)))
    if modulo:
        indices_validos = [i for i, d in enumerate(docs) if d["modulo"] == modulo]

    # pegar só os scores desses índices
    scores_filtrados = np.array([scores[i] for i in indices_validos])
    top_idx_local = scores_filtrados.argsort()[::-1][:top_k]

    resultados = []
    for local in top_idx_local:
        idx_global = indices_validos[local]
        d = docs[idx_global]
        resultados.append({
            "score": float(scores[idx_global]),
            "arquivo": d["arquivo"],
            "modulo": d["modulo"],
            "offset": d["offset"],
            "texto": d["texto"],
        })
    return resultados


In [ ]:
resultados = buscar_trechos(
    "screen capture core pipeline",
    modulo="screenpipe-core",
    top_k=3
)

for r in resultados:
    print("="*80)
    print("Arquivo:", r["arquivo"])
    print("Módulo:", r["modulo"], "| score:", round(r["score"], 3))
    print(r["texto"][:800])


In [ ]:
resultados = buscar_trechos("core pipeline orchestration", modulo="screenpipe-core", top_k=3)

trecho_combinado = "\n\n".join([r["texto"] for r in resultados])

print(trecho_combinado)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model2_id = "bigcode/starcoder2-7b"  # modelo de código

print("Carregando modelo 2:", model2_id)

tokenizer2 = AutoTokenizer.from_pretrained(model2_id)
model2 = AutoModelForCausalLM.from_pretrained(
    model2_id,
    torch_dtype=torch.float16,   # se não tiver GPU, troque para torch.float32 e tire isso
    device_map="auto"
)


In [ ]:
def run_arch_analysis_texto(texto, question_extra=""):
    """
    Recebe um texto (ex: concat de vários trechos do repo) e gera
    uma análise arquitetural usando o modelo 2 (StarCoder2).
    """
    # limitar tamanho pra não estourar contexto
    texto = texto[:4000]

    prompt = f"""
Você é um arquiteto de software especializado em Rust, TypeScript e sistemas multimodais.

Analise o seguinte trecho do projeto Screenpipe e responda:

1. Qual é a responsabilidade principal deste(s) módulo(s)?
2. Em que camada(s) da arquitetura Screenpipe ele(s) se encaixa(m)?
   - captura
   - processamento (OCR/STT)
   - armazenamento
   - APIs (Search/Memory/Streaming)
   - pipes/plugins (extensões)
3. Que outros componentes do projeto ele(s) provavelmente consome(m) ou expõe(m)?
4. {question_extra}

CÓDIGO / TRECHOS:
\"\"\"
/// Início dos trechos relevantes:
{texto}
\"\"\"


Agora produza uma análise arquitetural bem estruturada, em português:
"""

    inputs = tokenizer2(prompt, return_tensors="pt").to(model2.device)

    with torch.no_grad():
        outputs = model2.generate(
            **inputs,
            max_new_tokens=800,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer2.eos_token_id,
        )

    full_text = tokenizer2.decode(outputs[0], skip_special_tokens=True)
    resposta = full_text[len(prompt):].strip()
    return resposta


In [ ]:
def analisar_modulo_por_query(query, modulo=None, top_k=4, question_extra=""):
    resultados = buscar_trechos(query, modulo=modulo, top_k=top_k)

    # >>> ADICIONE ISTO <<<
    resultados = [r for r in resultados if len(r["texto"]) > 200]

    if not resultados:
        return "Nenhum trecho relevante ou suficientemente grande encontrado."

    partes = []
    for r in resultados:
        partes.append(r["texto"])

    texto = "\n\n".join(partes)[:4000]

    return run_arch_analysis_texto(texto, question_extra)


In [ ]:
resultados = [r for r in resultados if len(r["texto"]) > 200]


In [ ]:
with open("screenpipe-core/src/lib.rs") as f:
    code = f.read()

print(run_arch_analysis_texto(code[:4000], "Explique o papel do core."))


In [ ]:
tops = buscar_trechos("ffmpeg", modulo="screenpipe-core", top_k=4)
for r in tops:
    print("="*60)
    print(r["arquivo"])
    print(r["texto"][:500])


In [ ]:
#Análise Core
analise_core = analisar_modulo_por_query(
    query="capture screen audio pipeline orchestrator",
    modulo="screenpipe-core",
    top_k=4,
    question_extra="Diga se esse conjunto de trechos sugere um padrão microkernel/plugin ou pipe-and-filter, e justifique."
)

print(analise_core)
